In [1]:
"""Split a patent sheet into its separate figures.

Optimized for massive stitched canvases and text label absorption.
"""
import cv2 as cv
import numpy as np

def area(b):
    return max(0, b[2] - b[0]) * max(0, b[3] - b[1])

def frac_inside_padded(piece, group_box, pad_px):
    """
    Calculates how much of 'piece' falls inside the 'group' 
    if the group had an invisible padded halo around it.
    """
    hx1 = group_box[0] - pad_px
    hy1 = group_box[1] - pad_px
    hx2 = group_box[2] + pad_px
    hy2 = group_box[3] + pad_px
    
    iw = min(piece[2], hx2) - max(piece[0], hx1)
    ih = min(piece[3], hy2) - max(piece[1], hy1)
    
    if iw <= 0 or ih <= 0:
        return 0.0
    return (iw * ih) / area(piece)

# --- UPDATED PARAMETERS FOR STITCHED SHEETS ---
MIN_AREA  = 20       # Drop microscopic dust/speckles
CLOSE_PX  = 5        # Light morphological close to bridge dashed lines
PAD_PX    = 60       # "Halo" distance to pull in external labels ("Fig 1", "a")
CONTAIN   = 0.60     # Piece must be 60% inside the *halo* to merge
FIG_FRAC  = 0.004    # Dropped from 2% to 0.4% because stitched sheets are massive

def find_figures(ink, min_area=MIN_AREA, close_px=CLOSE_PX, pad_px=PAD_PX, contain=CONTAIN):
    """ink: uint8 image, ink = 255. Returns a list of [x1, y1, x2, y2]."""
    
    # 1. Light closing to bridge dashed lines internally before finding components
    if close_px > 1:
        kernel = cv.getStructuringElement(cv.MORPH_RECT, (close_px, close_px))
        work = cv.morphologyEx(ink, cv.MORPH_CLOSE, kernel)
    else:
        work = ink

    _, _, stats, _ = cv.connectedComponentsWithStats(work, connectivity=8)
    
    # Format: [x1, y1, x2, y2, area]
    parts = [[int(x), int(y), int(x + w), int(y + h), int(a)]
             for x, y, w, h, a in stats[1:] if a >= min_area]

    groups = [] # Store as dictionaries to track bounding box AND accumulated area
    
    # 2. Biggest first = seeds
    for p in sorted(parts, key=lambda x: x[4], reverse=True):
        pbox = p[0:4]
        parea = p[4]
        
        best_host = None
        best_overlap = -1
        
        # Find the best existing group to absorb this piece
        for g in groups:
            # GUARDRAIL: "When in doubt, separate." 
            # Never merge two massive seeds together just because their halos touch.
            # The piece being absorbed must be < 25% of the host's mass.
            if parea > g['area'] * 0.25:
                continue
                
            overlap = frac_inside_padded(pbox, g['box'], pad_px)
            if overlap >= contain and overlap > best_overlap:
                best_overlap = overlap
                best_host = g
                
        # 3. Merge or create new seed
        if best_host is not None:
            hx1, hy1, hx2, hy2 = best_host['box']
            # Expand the host's permanent bounding box to encapsulate the new piece
            best_host['box'] = [
                min(hx1, pbox[0]), min(hy1, pbox[1]),
                max(hx2, pbox[2]), max(hy2, pbox[3])
            ]
            best_host['area'] += parea
        else:
            groups.append({'box': pbox, 'area': parea})
            
    return [g['box'] for g in groups]

In [2]:
import glob
from pathlib import Path as PATH
import os

def ink_based_connected_components(src, debug_out):
    gray = cv.imread(src, cv.IMREAD_GRAYSCALE)
    _, ink = cv.threshold(gray, 128, 255, cv.THRESH_BINARY_INV)
    H, W = ink.shape
    
    min_fig_area = FIG_FRAC * W * H
    
    raw_figures = find_figures(ink)
    figures = sorted((b for b in raw_figures if area(b) >= min_fig_area),
                     key=lambda b: (b[1], b[0]))            # reading order
                     
    debug = cv.cvtColor(ink, cv.COLOR_GRAY2BGR)
    for i, (x1, y1, x2, y2) in enumerate(figures, 1):
        cv.rectangle(debug, (x1, y1), (x2 - 1, y2 - 1), (0, 255, 0), 2)
        
    cv.imwrite(debug_out, debug)

In [3]:
import cv2 as cv
import numpy as np
import glob
from pathlib import Path as PATH
import os

# =============================================================================
# 1. CORE MATH & FIGURE FINDING LOGIC
# =============================================================================

def area(b):
    return max(0, b[2] - b[0]) * max(0, b[3] - b[1])

def frac_inside_padded(piece, group_box, pad_px):
    """Calculates overlap if the group had an invisible padded halo around it."""
    hx1 = group_box[0] - pad_px
    hy1 = group_box[1] - pad_px
    hx2 = group_box[2] + pad_px
    hy2 = group_box[3] + pad_px
    
    iw = min(piece[2], hx2) - max(piece[0], hx1)
    ih = min(piece[3], hy2) - max(piece[1], hy1)
    
    if iw <= 0 or ih <= 0:
        return 0.0
    return (iw * ih) / area(piece)

# --- UPDATED PARAMETERS FOR STITCHED SHEETS ---
MIN_AREA  = 20       # Drop microscopic dust/speckles
CLOSE_PX  = 5        # Light morphological close to bridge dashed lines
PAD_PX    = 60       # "Halo" distance to pull in external labels ("Fig 1", "a")
CONTAIN   = 0.60     # Piece must be 60% inside the *halo* to merge
FIG_FRAC  = 0.004    # 0.4% area threshold for massive stitched sheets

def find_figures(ink, min_area=MIN_AREA, close_px=CLOSE_PX, pad_px=PAD_PX, contain=CONTAIN):
    if close_px > 1:
        kernel = cv.getStructuringElement(cv.MORPH_RECT, (close_px, close_px))
        work = cv.morphologyEx(ink, cv.MORPH_CLOSE, kernel)
    else:
        work = ink

    _, _, stats, _ = cv.connectedComponentsWithStats(work, connectivity=8)
    
    parts = [[int(x), int(y), int(x + w), int(y + h), int(a)]
             for x, y, w, h, a in stats[1:] if a >= min_area]

    groups = [] 
    
    for p in sorted(parts, key=lambda x: x[4], reverse=True):
        pbox = p[0:4]
        parea = p[4]
        
        best_host = None
        best_overlap = -1
        
        for g in groups:
            # GUARDRAIL: Never merge two massive seeds together.
            if parea > g['area'] * 0.25:
                continue
                
            overlap = frac_inside_padded(pbox, g['box'], pad_px)
            if overlap >= contain and overlap > best_overlap:
                best_overlap = overlap
                best_host = g
                
        if best_host is not None:
            hx1, hy1, hx2, hy2 = best_host['box']
            best_host['box'] = [
                min(hx1, pbox[0]), min(hy1, pbox[1]),
                max(hx2, pbox[2]), max(hy2, pbox[3])
            ]
            best_host['area'] += parea
        else:
            groups.append({'box': pbox, 'area': parea})
            
    return [g['box'] for g in groups]

# =============================================================================
# 2. SORTING & DRAWING LOGIC
# =============================================================================

def reading_order(boxes, min_fig_area):
    """Filters out tiny noise boxes and sorts top-to-bottom, left-to-right."""
    valid_boxes = [b for b in boxes if area(b) >= min_fig_area]
    return sorted(valid_boxes, key=lambda b: (b[1], b[0]))

def ink_based_connected_components(src, debug_out):
    gray = cv.imread(src, cv.IMREAD_GRAYSCALE)
    _, ink = cv.threshold(gray, 128, 255, cv.THRESH_BINARY_INV)
    
    H, W = ink.shape
    min_fig_area = FIG_FRAC * W * H
    
    # Apply finding logic, filter by scale, and sort
    figures = reading_order(find_figures(ink), min_fig_area)
    
    debug = cv.cvtColor(ink, cv.COLOR_GRAY2BGR)
    for i, (x1, y1, x2, y2) in enumerate(figures, 1):
        cv.rectangle(debug, (x1, y1), (x2 - 1, y2 - 1), (0, 255, 0), 2)   # corners are inclusive
        cv.putText(debug, str(i), (x1 + 8, y1 + 40), cv.FONT_HERSHEY_SIMPLEX, 1.4, (0, 255, 0), 3)
        
    # uncomment the following lines to save each figure as a separate image
    #     cv.imwrite(f"figure_{i}.png", gray[y1:y2, x1:x2])
    cv.imwrite(debug_out, debug)
    # print(f"{len(figures)} figures:", figures)

# =============================================================================
# 3. EXECUTION
# =============================================================================

directory_path = "../../data/figure_extractor_test/"
all_sketch_pages = glob.glob(directory_path + "*.png")
debug_directory = PATH("../../results/figure_extractor_v3_test_output/")
debug_directory.mkdir(parents=True, exist_ok=True)

for src in all_sketch_pages:
    src_filename = src.split("/")[-1]
    debug_out = os.path.join(debug_directory, PATH(f"debug_{src_filename}"))
    print(f"Debug output saved to: {debug_out}")
    ink_based_connected_components(src, debug_out)

Debug output saved to: ../../results/figure_extractor_v3_test_output/debug_2393_2398_page_3.png
Debug output saved to: ../../results/figure_extractor_v3_test_output/debug_2424_2429_page_2.png
Debug output saved to: ../../results/figure_extractor_v3_test_output/debug_2424_2429_page_3.png
Debug output saved to: ../../results/figure_extractor_v3_test_output/debug_2398_2403_page_3.png
Debug output saved to: ../../results/figure_extractor_v3_test_output/debug_sheet_pages_0003_0005.png
Debug output saved to: ../../results/figure_extractor_v3_test_output/debug_2398_2403_page_4.png
Debug output saved to: ../../results/figure_extractor_v3_test_output/debug_sheet_pages_0010_0012.png
Debug output saved to: ../../results/figure_extractor_v3_test_output/debug_sheet_pages_0008_0009.png
Debug output saved to: ../../results/figure_extractor_v3_test_output/debug_2453_2458_page_3.png
Debug output saved to: ../../results/figure_extractor_v3_test_output/debug_2394_2399_page_2.png
Debug output saved to: ..